# Backbone Parameter Counts

This notebook builds parameter-count tables from `configs/backbones.yaml` using `models/parameter_count.py`. The ViT-family rows are architecture-level encoder counts computed from the configured ViT specifications. LTX-Video rows are exact transformer parameter counts derived from Hugging Face safetensors index metadata for the configured 2B and 13B variants.

For an already-loaded `torch.nn.Module` or adapter, use `count_module_parameters(...)` or `count_adapter_parameters(...)` from the same helper module.


In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'configs' / 'backbones.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not find configs/backbones.yaml from the current notebook path.')


REPO_ROOT = find_repo_root(Path.cwd())
PARAMETER_COUNT_PATH = REPO_ROOT / 'models' / 'parameter_count.py'
spec = importlib.util.spec_from_file_location('probe4physics_parameter_count', PARAMETER_COUNT_PATH)
parameter_count = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parameter_count
assert spec.loader is not None
spec.loader.exec_module(parameter_count)

build_vit_parameter_table = parameter_count.build_vit_parameter_table
select_vit_size_comparison_rows = parameter_count.select_vit_size_comparison_rows
format_parameter_count = parameter_count.format_parameter_count

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)


def display_parameter_count(value):
    if pd.isna(value):
        return 'NA'
    value = float(value)
    if value >= 1_000_000_000:
        return f'{value / 1_000_000_000:.2f}B'
    return format_parameter_count(value)


rows = build_vit_parameter_table(config_path=REPO_ROOT / 'configs' / 'backbones.yaml')
all_counts = pd.DataFrame(rows).sort_values(['backbone', 'total_parameters', 'variant']).reset_index(drop=True)
all_counts['count_source'] = 'computed ViT encoder formula'
all_counts['source_note'] = 'architecture-level encoder count from configs/backbones.yaml'
all_counts['hf_model_id'] = pd.NA
all_counts['params'] = all_counts['total_parameters'].map(display_parameter_count)
all_counts['fixed_params'] = all_counts['fixed_parameters'].map(display_parameter_count)

comparison = pd.DataFrame(select_vit_size_comparison_rows(rows))
extra_summary_variants = [
    ('videomae_v2', 'vit_base_16_224'),
]
extra_summary_rows = all_counts[
    all_counts.set_index(['backbone', 'variant']).index.isin(extra_summary_variants)
]
comparison = pd.concat([comparison, extra_summary_rows], ignore_index=True, sort=False)
comparison = comparison.drop_duplicates(['backbone', 'variant'], keep='first')
comparison = comparison.sort_values(['backbone', 'total_parameters', 'variant']).reset_index(drop=True)
comparison['count_source'] = comparison['count_source'].fillna('computed ViT encoder formula')
comparison['source_note'] = comparison['source_note'].fillna('architecture-level encoder count from configs/backbones.yaml')
comparison['hf_model_id'] = comparison['hf_model_id'].astype('object').where(comparison['hf_model_id'].notna(), pd.NA)
comparison['params'] = comparison['total_parameters'].map(display_parameter_count)
comparison['fixed_params'] = comparison['fixed_parameters'].map(display_parameter_count)

# LTX counts are for the diffusion transformer component used as the feature backbone.
# They come from transformer/diffusion_pytorch_model.safetensors.index.json metadata.
ltx_counts = pd.DataFrame([
    {
        'backbone': 'ltx_video',
        'variant': 'ltx_2b_0_9_8_distilled',
        'size_label': 'LTX-2B',
        'model_name': 'ltx_transformer_28',
        'is_default_variant': False,
        'crop_size': 224,
        'frames_per_clip': 16,
        'patch_size': 1,
        'tubelet_size': pd.NA,
        'depth': 28,
        'embed_dim': pd.NA,
        'num_heads': pd.NA,
        'mlp_ratio': pd.NA,
        'total_parameters': 1_923_385_472,
        'total_millions': 1_923.385472,
        'fixed_parameters': 1_923_385_472,
        'formula': 'transformer_safetensors_index_total_size / 4 bytes_per_fp32_param',
        'count_source': 'exact transformer safetensors metadata',
        'source_note': 'Lightricks/LTX-Video transformer index total_size=7,693,541,888 bytes, F32 tensors.',
        'hf_model_id': 'Lightricks/LTX-Video',
    },
    {
        'backbone': 'ltx_video',
        'variant': 'ltxv_13b_0_9_8_distilled',
        'size_label': 'LTX-13B',
        'model_name': 'ltx_transformer_48',
        'is_default_variant': True,
        'crop_size': 224,
        'frames_per_clip': 16,
        'patch_size': 1,
        'tubelet_size': pd.NA,
        'depth': 48,
        'embed_dim': pd.NA,
        'num_heads': pd.NA,
        'mlp_ratio': pd.NA,
        'total_parameters': 13_042_569_344,
        'total_millions': 13_042.569344,
        'fixed_parameters': 13_042_569_344,
        'formula': 'transformer_safetensors_index_total_size / 2 bytes_per_bf16_param',
        'count_source': 'exact transformer safetensors metadata',
        'source_note': 'Lightricks/LTX-Video-0.9.8-13B-distilled transformer index total_size=26,085,138,688 bytes, BF16 tensors.',
        'hf_model_id': 'Lightricks/LTX-Video-0.9.8-13B-distilled',
    },
])
ltx_counts['params'] = ltx_counts['total_parameters'].map(display_parameter_count)
ltx_counts['fixed_params'] = ltx_counts['fixed_parameters'].map(display_parameter_count)

comparison_with_ltx = pd.concat([comparison, ltx_counts], ignore_index=True, sort=False)
comparison_with_ltx = comparison_with_ltx.sort_values(
    ['backbone', 'total_parameters', 'variant'],
    kind='stable',
).reset_index(drop=True)


## Selected backbone variants

This summary table includes ViT-L rows, the largest configured ViT variant for each family, the VideoMAE v2 ViT-B backbone used in the ablation, and the configured LTX transformer variants.


In [ ]:
summary_columns = [
    'backbone',
    'variant',
    'size_label',
    'model_name',
    'count_source',
    'is_default_variant',
    'crop_size',
    'frames_per_clip',
    'depth',
    'embed_dim',
    'num_heads',
    'params',
    'total_parameters',
    'fixed_params',
    'formula',
]

comparison_with_ltx[summary_columns].style.format({'total_parameters': '{:,}'})


## Detailed ViT component breakdown

This detailed component table is kept to ViT-family variants only. LTX rows are reported separately below because the LTX diffusion transformer is counted from checkpoint metadata, not decomposed by the ViT encoder formula.


In [ ]:
breakdown_columns = [
    'backbone',
    'variant',
    'size_label',
    'model_name',
    'patch_embed_params',
    'patch_embed_img_params',
    'position_embedding_params',
    'blocks_attention_params',
    'blocks_mlp_params',
    'blocks_norm_params',
    'blocks_total_params',
    'final_norm_params',
    'hierarchical_norm_params',
    'modality_embedding_params',
    'total_parameters',
]

all_counts[breakdown_columns].style.format({column: '{:,}' for column in breakdown_columns if column.endswith('_params') or column == 'total_parameters'})

## LTX transformer counts

These rows use exact parameter counts from the configured LTX transformer safetensors index metadata. The scope is the frozen LTX diffusion transformer used for probing, excluding text encoder, tokenizer, scheduler, and VAE components.


In [ ]:
ltx_columns = [
    'backbone',
    'variant',
    'size_label',
    'model_name',
    'hf_model_id',
    'depth',
    'params',
    'total_parameters',
    'count_source',
    'source_note',
]

ltx_counts[ltx_columns].style.format({'total_parameters': '{:,}'})


## Probed Layer Indices

This table reports the configured probe-layer indices for each model/backbone variant in the parameter-count tables. ViT-family rows use 1-based transformer block ids. LTX rows use the flattened `probe.layer` slot ids, with the corresponding transformer block ids shown separately because each block is repeated across diffusion noise levels.


In [ ]:
def resolve_configured_probe_layers(row):
    backbone = row['backbone']
    model_name = row['model_name']
    depth = int(row['depth'])
    relative_depths = (0.25, 0.5, 0.75, 1.0)

    if backbone == 'jepa_v2_1':
        hierarchical_layers = {
            12: (3, 6, 9, 12),
            24: (6, 12, 18, 24),
            40: (10, 20, 30, 40),
            48: (12, 24, 38, 48),
        }
        return hierarchical_layers[depth]

    resolved = []
    for value in relative_depths:
        layer_id = int(round(depth * value))
        layer_id = max(1, min(depth, layer_id))
        if layer_id not in resolved:
            resolved.append(layer_id)
    return tuple(resolved)


def format_layer_tuple(values):
    return ', '.join(str(int(value)) for value in values)


layer_rows = []
layer_source = pd.concat([all_counts, ltx_counts], ignore_index=True, sort=False)
layer_source = layer_source.sort_values(['backbone', 'total_parameters', 'variant'], kind='stable').reset_index(drop=True)

for row in layer_source.to_dict('records'):
    if row['backbone'] == 'ltx_video':
        block_layers = resolve_configured_probe_layers(row)
        noise_levels = (1, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1)
        probe_slots = tuple(range(1, len(block_layers) * len(noise_levels) + 1))
        probed_layer_indices = f"{probe_slots[0]}-{probe_slots[-1]}"
        layer_note = f"10 noise levels x blocks {format_layer_tuple(block_layers)}"
    else:
        block_layers = resolve_configured_probe_layers(row)
        probed_layer_indices = format_layer_tuple(block_layers)
        layer_note = '1-based transformer block ids'

    layer_rows.append({
        'model': row['backbone'],
        'backbone': row['size_label'],
        'variant': row['variant'],
        'depth': row['depth'],
        'probed_layer_indices': probed_layer_indices,
        'layer_note': layer_note,
    })

probed_layers = pd.DataFrame(layer_rows)
probed_layers
